In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')

# Servers table
servers_inventory = {
    "Host_ID": ["SRV-01", "SRV-02", "SRV-03"],
    "Role": ["Web Front", "API Gateway", "Database Replica"]
}

# Interfaces table
live_interfaces = {
    "Interface_ID": ["eth0", "eth1"],
    "Mapped_Host": ["SRV-01", "SRV-02"],
    "IP_Address": ["10.0.0.4", "10.0.0.9"]
}

df_srv = pd.DataFrame(servers_inventory)
df_inf = pd.DataFrame(live_interfaces)

df_srv.to_sql('Servers', conn, index=False, if_exists='replace')
df_inf.to_sql('Interfaces', conn, index=False, if_exists='replace')

#Broken Query Framework 
faulty_query = """
SELECT s.Host_ID, s.Role, i.Interface_ID, i.IP_Address 
FROM Servers s 
INNER JOIN Interfaces i ON s.Host_ID = i.Mapped_Host;
"""
print("--- Faulty Output Metrics (Missing Database Replica SRV-03) ---") 
pd.read_sql_query(faulty_query, conn)


--- Faulty Output Metrics (Missing Database Replica SRV-03) ---


,Host_ID,Role,Interface_ID,IP_Address
0,SRV-01,Web Front,eth0,10.0.0.4
1,SRV-02,API Gateway,eth1,10.0.0.9


In [ ]:
#1. Explain why SRV-03 is excluded from the query framework above based on the mechanics of an INNER JOIN.
An INNER JOIN only returns rows where a matching record exists in both tables.
Since SRV-03 (server_id = 3) has no corresponding record in live_interfaces, 
Therefore, the INNER JOIN excludes SRV-03 from the result set.

In [3]:
#2. Rewrite the query below to implement a join strategy 
# that outputs all server configurations from inventory, regardless of
#  whether they have a live network interface.
print("""Use a LEFT JOIN instead of an INNER JOIN:
A LEFT JOIN returns all rows from inventory, 
even when there is no matching record in network_interfaces. 
In those cases, the columns from the right table are returned as NULL, allowing SRV-03 to appear in the output.""")
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')

# Servers table
servers_inventory = {
    "Host_ID": ["SRV-01", "SRV-02", "SRV-03"],
    "Role": ["Web Front", "API Gateway", "Database Replica"]
}

# Interfaces table
live_interfaces = {
    "Interface_ID": ["eth0", "eth1"],
    "Mapped_Host": ["SRV-01", "SRV-02"],
    "IP_Address": ["10.0.0.4", "10.0.0.9"]
}

df_srv = pd.DataFrame(servers_inventory)
df_inf = pd.DataFrame(live_interfaces)

df_srv.to_sql('Servers', conn, index=False, if_exists='replace')
df_inf.to_sql('Interfaces', conn, index=False, if_exists='replace')

# Correct query using LEFT JOIN
query = """
SELECT
    s.Host_ID,
    s.Role,
    i.Interface_ID,
    i.IP_Address
FROM Servers s
LEFT JOIN Interfaces i
ON s.Host_ID = i.Mapped_Host;
"""

result = pd.read_sql_query(query, conn)
print(result)

Use a LEFT JOIN instead of an INNER JOIN:
A LEFT JOIN returns all rows from inventory, 
even when there is no matching record in network_interfaces. 
In those cases, the columns from the right table are returned as NULL, allowing SRV-03 to appear in the output.
  Host_ID              Role Interface_ID IP_Address
0  SRV-01         Web Front         eth0   10.0.0.4
1  SRV-02       API Gateway         eth1   10.0.0.9
2  SRV-03  Database Replica          NaN        NaN
